# 🔬 Datenanalyse-Agent mit LangGraph – Stufe 1
---

## Überblick

Dieses Notebook implementiert einen **agentischen Datenanalyse-Assistenten**, 
der eine CSV- oder Excel-Datei entgegennimmt und autonom eine explorative 
Datenanalyse durchführt.

### Architektur (Stufe 1 – linearer Graph mit ReAct-Zyklus)

```
START → data_profiler → analyst ⇄ tools → END
```

| Knoten | Typ | Aufgabe |
|--------|-----|---------|
| **Data Profiler** | Deterministisch (kein LLM) | Datei einlesen, Struktur erfassen, Statistiken berechnen |
| **Analyst** | LLM + Tool-Calling | Analyseplan erstellen, Python-Code generieren & ausführen |
| **Tools** | Code-Sandbox | Python-Code sicher ausführen (pandas, matplotlib, seaborn) |

### Was wir hier lernen
- `StateGraph` mit `TypedDict` definieren
- Knoten als Python-Funktionen implementieren
- Feste Kanten (`add_edge`) vs. konditionale Kanten (`add_conditional_edges`)
- Den **ReAct-Zyklus**: LLM entscheidet selbst, wann es Tools aufruft
- Gradio-Interface als einfache Benutzeroberfläche

### Warum LangGraph statt LangChain?
Der **Zyklus** `analyst → tools → analyst` ist mit LangChain (LCEL-Chains) 
nicht möglich, weil Chains nur DAGs (Directed Acyclic Graphs) erlauben – 
keine Rückwärtskanten. LangGraph erlaubt Zyklen, weil es ein 
zustandsbehafteter Graph ist.

## 1. Installation & Setup

In [ ]:
# Abhängigkeiten installieren (einmalig ausführen)
# !pip install langgraph langchain-openai pandas openpyxl matplotlib seaborn gradio

In [ ]:
import os
import sys
import json
import io
import contextlib
from typing import TypedDict, Annotated

import pandas as pd
import matplotlib
matplotlib.use("Agg")  # Backend ohne GUI – wichtig für Server-Betrieb
import matplotlib.pyplot as plt
import seaborn as sns
import getpass

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

print("✓ Alle Bibliotheken geladen")

In [ ]:
# ══════════════════════════════════════════════════════════════
# API-Key setzen
# ══════════════════════════════════════════════════════════════
DEEPINFRA_API_KEY = getpass.getpass("DeepInfra API-Key eingeben: ")

llm = ChatOpenAI(
    #model_name="Qwen/Qwen3-Max",
    #model_name="nvidia/Nemotron-3-Nano-Omni-30B-A3B-Reasoning",
    model_name="Qwen/Qwen3.6-35B-A3B",
    openai_api_key=DEEPINFRA_API_KEY,
    openai_api_base="https://api.deepinfra.com/v1/openai",
    max_tokens=5000,
    temperature=0,  # Für RAG: keine Kreativität, sondern Fakten
)

print("✓ API-Key gefunden")

## 2. State definieren

Der **State** ist das zentrale Konzept in LangGraph. Er ist ein `TypedDict`, 
der alle Daten enthält, die zwischen den Knoten fließen.

**Wichtiges Detail:** Das `Annotated[list, add_messages]` bei `messages` 
ist ein sogenannter **Reducer**. Er sagt LangGraph: „Wenn ein Knoten 
neue Nachrichten zurückgibt, hänge sie an die bestehende Liste an, 
statt sie zu überschreiben." Ohne den Reducer würde jeder Knoten 
die gesamte Konversationshistorie löschen.

In [ ]:
class AnalysisState(TypedDict):
    """
    State-Schema für den Datenanalyse-Agenten.
    Jedes Feld ist ein Kanal (Channel), über den Knoten kommunizieren.
    """
    # Eingabe: Pfad zur Datendatei
    file_path: str
    
    # Data Profiler → Analyst: strukturiertes Datenprofil
    data_profile: dict
    
    # Analyst ⇄ Tools: LLM-Konversation mit Tool-Calls und -Ergebnissen
    # Der Reducer add_messages sorgt dafür, dass Nachrichten
    # angehängt statt überschrieben werden
    messages: Annotated[list, add_messages]

## 3. Tool definieren: Python-Sandbox

Der Analyst-Agent braucht die Fähigkeit, Python-Code auszuführen.
Wir definieren dafür ein **Tool** – eine Funktion, die der LLM 
eigenständig aufrufen kann.

**Sicherheitshinweis:** In einer Produktionsumgebung würde man den 
Code in einem Docker-Container oder einer E2B-Sandbox ausführen. 
Für unser Lernprojekt reicht der eingeschränkte Namespace.

In [ ]:
@tool
def execute_python(code: str) -> str:
    """Führt Python-Code aus und gibt stdout + stderr zurück.
    
    Verfügbare Bibliotheken: pandas (als pd), matplotlib.pyplot (als plt),
    seaborn (als sns), numpy (als np).
    
    WICHTIG: Alle Plots mit plt.savefig('plots/plot_name.png') speichern,
    NICHT plt.show() verwenden. Print-Ausgaben werden zurückgegeben.
    """
    import numpy as np
    
    # Eingeschränkter Namespace – nur sichere Bibliotheken
    namespace = {
        "pd": pd,
        "np": np,
        "plt": plt,
        "sns": sns,
    }

    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()

    try:
        with contextlib.redirect_stdout(stdout_capture), \
             contextlib.redirect_stderr(stderr_capture):
            exec(code, namespace)

        output = stdout_capture.getvalue()
        errors = stderr_capture.getvalue()
        result = output if output else "(Kein Output – nutze print())"
        if errors:
            result += f"\n\nWarnungen:\n{errors}"
        return result

    except Exception as e:
        return f"FEHLER:\n{type(e).__name__}: {e}"

print("✓ Tool 'execute_python' definiert")

## 4. Knoten definieren

Jeder Knoten ist eine normale Python-Funktion mit der Signatur:

```python
def knoten_name(state: AnalysisState) -> dict:
    # ... Logik ...
    return {"feld": neuer_wert}  # Teilupdate des State
```

Der Rückgabewert ist ein **Teilupdate** – man gibt nur die Felder 
zurück, die sich geändert haben.

### 4a. Data Profiler (deterministisch – kein LLM)

Dieser Knoten zeigt eine wichtige Architekturlektion: **Nicht alles 
muss ein LLM sein.** Das Datenprofil ist rein mechanisch (Pandas-Statistiken), 
kein Token fließt durch ein Sprachmodell. LLM-Calls sind teuer und 
langsam – sie gehören nur dorthin, wo Urteilsvermögen gefragt ist.

In [ ]:
def data_profiler(state: AnalysisState) -> dict:
    """
    Knoten 1: Liest die Datei ein und erstellt ein strukturiertes Profil.
    Dieses Profil wird dem LLM als Kontext übergeben, damit er die
    Datenstruktur versteht, ohne die Rohdaten zu sehen.
    """
    file_path = state["file_path"]
    
    # Datei einlesen (CSV oder Excel)
    if file_path.endswith((".xlsx", ".xls")):
        df = pd.read_excel(file_path)
    else:
        try:
            df = pd.read_csv(file_path)
        except Exception:
            df = pd.read_csv(file_path, sep=";")
    
    # Profil zusammenstellen
    profile = {
        "dateiname": os.path.basename(file_path),
        "zeilen": int(df.shape[0]),
        "spalten": int(df.shape[1]),
        "spaltennamen": list(df.columns),
        "datentypen": {
            col: str(dtype) for col, dtype in df.dtypes.items()
        },
        "fehlende_werte": {
            col: int(count) 
            for col, count in df.isnull().sum().items() 
            if count > 0
        },
        "numerische_spalten": list(
            df.select_dtypes(include="number").columns
        ),
        "kategorische_spalten": list(
            df.select_dtypes(include=["object", "category"]).columns
        ),
        "grundstatistik": json.loads(
            df.describe(include="all").to_json()
        ),
        "stichprobe_5_zeilen": json.loads(
            df.head().to_json(orient="records", date_format="iso")
        ),
    }
    
    return {"data_profile": profile}

print("✓ Knoten 'data_profiler' definiert")

### 4b. Analyst (LLM mit Tool-Calling / ReAct-Pattern)

Der Analyst bekommt das Datenprofil und entscheidet **selbstständig**:
- Welche Analysen sinnvoll sind
- Welchen Python-Code er dafür generiert
- Ob er weitere Analysen durchführen will (→ erneuter Tool-Call) 
  oder ob er fertig ist (→ Textantwort ohne Tool-Call)

Das ist der **ReAct-Zyklus** (Reason + Act): Der LLM steuert den 
Kontrollfluss des Graphen.

In [ ]:
def analyst(state: AnalysisState) -> dict:
    """
    Knoten 2: LLM-Agent, der Datenanalysen plant und durchführt.
    """
    profile = state["data_profile"]
    file_path = state["file_path"]
    
    system_prompt = f"""Du bist ein erfahrener Datenanalyst. Du erhältst ein 
Datenprofil und führst eine explorative Datenanalyse durch.

DATENPROFIL:
{json.dumps(profile, indent=2, ensure_ascii=False)}

DEINE AUFGABEN:
1. Lies die Datei ein: df = pd.read_csv("{file_path}")
   (bzw. pd.read_excel() für Excel-Dateien)
2. Führe 2-3 sinnvolle Analysen durch, passend zu den Datentypen:
   - Numerische Spalten → Verteilungen, Korrelationen, Ausreißer
   - Kategorische Spalten → Häufigkeiten, Gruppierungen
   - Zeitliche Spalten → Trends, saisonale Muster
   - Fehlende Werte → Muster erkennen, Umgang empfehlen
3. Erstelle zu jeder Analyse eine Visualisierung
4. Fasse am Ende deine Erkenntnisse zusammen (auf Deutsch)

REGELN FÜR CODE:
- plt.savefig('plots/plot_name.png', dpi=150, bbox_inches='tight') verwenden
- Danach plt.close() aufrufen (verhindert Speicherlecks)
- Ergebnisse immer mit print() ausgeben
- Pro Tool-Aufruf EIN thematisch geschlossenes Code-Stück
- Deutsche Beschriftungen in Plots"""

    # Beim ersten Aufruf: System-Prompt + Startanweisung setzen
    if not state.get("messages"):
        initial_messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content="Bitte analysiere den Datensatz."),
        ]
        llm_with_tools = llm.bind_tools([execute_python])
        response = llm_with_tools.invoke(initial_messages)
        # Alle drei Messages zurückgeben — der Reducer hängt sie an
        return {"messages": initial_messages + [response]}
    else:
        llm_with_tools = llm.bind_tools([execute_python])
        response = llm_with_tools.invoke(state["messages"])
        # Nur die neue Antwort — der Reducer hängt sie an die bestehende Historie
        return {"messages": [response]}

print("✓ Knoten 'analyst' definiert")

## 5. Konditionale Kante: Tool aufrufen oder fertig?

Diese Funktion ist das **Routing** des ReAct-Zyklus. Sie prüft 
die letzte Nachricht des LLM:
- Enthält sie Tool-Calls? → Weiter zum `tools`-Knoten
- Keine Tool-Calls? → Der Agent ist fertig → `END`

In [ ]:
def should_use_tool(state: AnalysisState) -> str:
    """
    Konditionale Kante: Prüft, ob der LLM ein Tool aufrufen will.
    
    Returns:
        "tools" → Code ausführen
        END     → Analyse abgeschlossen
    """
    last_message = state["messages"][-1]
    
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return END

print("✓ Routing-Funktion definiert")

## 6. Graph zusammenbauen

Hier kommt alles zusammen. Der `StateGraph` verbindet Knoten 
mit Kanten — wie ein Flussdiagramm, das zur Laufzeit durchlaufen wird.

```
START ──edge──→ data_profiler ──edge──→ analyst
                                          │
                                    conditional_edge
                                          │
                                ┌─────────┴─────────┐
                                ▼                   ▼
                              tools               END
                                │
                              edge
                                │
                                └──→ analyst  (← Zyklus!)
```

Der **Zyklus** `analyst → tools → analyst` ist der Kern. 
LangChain-Chains sind DAGs — keine Rückwärtskanten. LangGraph 
erlaubt Zyklen, weil es ein zustandsbehafteter Graph ist.

In [ ]:
def build_graph():
    """Baut den LangGraph-Graphen und kompiliert ihn."""
    
    graph = StateGraph(AnalysisState)
    
    # ── Knoten registrieren ──
    graph.add_node("data_profiler", data_profiler)
    graph.add_node("analyst", analyst)
    graph.add_node("tools", ToolNode([execute_python]))
    
    # ── Feste Kanten ──
    graph.add_edge(START, "data_profiler")       # Start → Profiler
    graph.add_edge("data_profiler", "analyst")   # Profiler → Analyst
    graph.add_edge("tools", "analyst")           # Tools → zurück zum Analyst
    
    # ── Konditionale Kante ──
    graph.add_conditional_edges(
        "analyst",         # Von diesem Knoten...
        should_use_tool,   # ...entscheidet diese Funktion wohin
        path_map={"tools": "tools", END: END}   # ← das fehlt
    )
    
    return graph.compile()

# Graph bauen
app = build_graph()
print("✓ Graph kompiliert und bereit")

### Graph visualisieren (optional)

In [ ]:
import base64
from IPython import display

def display_graph(graph_app):
    """Zeigt den LangGraph als Mermaid-Diagramm an (lokal, ohne externen Dienst)."""
    try:
        # Methode 1: Direkt als PNG rendern (braucht graphviz/pygraphviz)
        png_data = graph_app.get_graph().draw_mermaid_png()
        display.display(display.Image(data=png_data))
    except Exception:
        # Fallback: Mermaid-Code als Text ausgeben
        mermaid_code = graph_app.get_graph().draw_mermaid()
        print(mermaid_code)


display_graph(app)

## 7. Ausführungsfunktion

Diese Funktion kapselt den gesamten Ablauf und formatiert die 
Ergebnisse als Markdown-Bericht. Sie wird vom Gradio-Interface aufgerufen.

In [ ]:
def run_analysis(file_path: str) -> str:
    """
    Führt die komplette Datenanalyse durch.
    
    Args:
        file_path: Pfad zur CSV- oder Excel-Datei
        
    Returns:
        Markdown-formatierter Analysebericht
    """
    app = build_graph()
    
    initial_state = {
        "file_path": file_path,
        "data_profile": {},
        "messages": [],
    }
    
    # Fortschritt sammeln
    log_lines = []
    code_blocks = []        # ← NEU: vollständige Code-Snippets sammeln
    final_text = ""
    
    for step in app.stream(initial_state, stream_mode="updates"):
        node_name = list(step.keys())[0]
        
        if node_name == "data_profiler":
            profile = step[node_name].get("data_profile", {})
            log_lines.append(
                f"📊 **Datenprofil erstellt:** "
                f"{profile.get('zeilen', '?')} Zeilen, "
                f"{profile.get('spalten', '?')} Spalten"
            )
            num = profile.get("numerische_spalten", [])
            kat = profile.get("kategorische_spalten", [])
            if num:
                log_lines.append(f"  - Numerisch: {', '.join(num)}")
            if kat:
                log_lines.append(f"  - Kategorisch: {', '.join(kat)}")
            if profile.get("fehlende_werte"):
                log_lines.append(
                    f"  - ⚠️ Fehlende Werte: {profile['fehlende_werte']}"
                )
        
        elif node_name == "analyst":
            msg = step[node_name]["messages"][-1]
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    full_code = tc["args"]["code"]
                    code_preview = full_code[:80].replace("\n", " ")
                    log_lines.append(f"🔧 Code: `{code_preview}...`")
                    code_blocks.append(full_code)    # ← NEU: vollständig speichern
            elif hasattr(msg, "content") and msg.content:
                final_text = msg.content
        
        elif node_name == "tools":
            for msg in step[node_name]["messages"]:
                if msg.content and not msg.content.startswith("(Kein"):
                    preview = msg.content[:120].replace("\n", " ")
                    log_lines.append(f"✅ Ergebnis: `{preview}`")
    
    # Bericht zusammensetzen
    report = "## 📋 Analyseverlauf\n\n"
    report += "\n\n".join(log_lines)
    report += "\n\n---\n\n"
    report += "## 📊 Ergebnisse & Erkenntnisse\n\n"
    report += final_text if final_text else "*(Keine Zusammenfassung generiert)*"
    
    # ── NEU: Code-Appendix ──
    if code_blocks:
        report += "\n\n---\n\n"
        report += "## 🐍 Verwendeter Analyse-Code\n\n"
        for i, code in enumerate(code_blocks, 1):
            report += f"### Schritt {i}\n\n"
            report += f"```python\n{code}\n```\n\n"
    
    return report

## 8. Beispieldaten erzeugen & Schnelltest

In [ ]:
# ── Beispieldaten erzeugen ──
beispiel_csv = "beispiel_verkaufsdaten.csv"

if not os.path.exists(beispiel_csv):
    daten = [
        "datum,produkt,kategorie,region,umsatz,menge,rabatt_prozent,kundenzufriedenheit",
        "2024-01-05,Widget Pro,Elektronik,Nord,1250.00,25,5,4.2",
        "2024-01-08,Widget Pro,Elektronik,Süd,980.50,18,10,3.8",
        "2024-01-12,Gadget Mini,Elektronik,West,2340.00,65,0,4.5",
        "2024-01-15,Office Chair,Möbel,Nord,890.00,10,15,4.0",
        "2024-01-18,Widget Pro,Elektronik,Ost,1100.00,22,5,4.1",
        "2024-01-22,Standing Desk,Möbel,Süd,3200.00,8,0,4.8",
        "2024-01-25,Gadget Mini,Elektronik,Nord,1890.50,52,8,4.3",
        "2024-02-01,Office Chair,Möbel,West,445.00,5,20,3.5",
        "2024-02-05,Widget Pro,Elektronik,Süd,2100.00,42,0,4.6",
        "2024-02-08,Kabelset,Zubehör,Ost,320.00,80,0,3.9",
        "2024-02-12,Standing Desk,Möbel,Nord,4800.00,12,5,4.7",
        "2024-02-15,Gadget Mini,Elektronik,West,1560.00,43,12,3.6",
        "2024-02-18,Widget Pro,Elektronik,Ost,870.00,16,15,3.4",
        "2024-02-22,Kabelset,Zubehör,Süd,480.00,120,0,4.0",
        "2024-02-25,Office Chair,Möbel,Nord,1780.00,20,8,4.4",
        "2024-03-01,Standing Desk,Möbel,West,1600.00,4,0,4.9",
        "2024-03-05,Gadget Mini,Elektronik,Nord,3120.00,87,5,4.2",
        "2024-03-08,Widget Pro,Elektronik,Süd,1650.00,33,10,3.7",
        "2024-03-12,Kabelset,Zubehör,Ost,560.00,140,3,4.1",
        "2024-03-15,Office Chair,Möbel,West,,15,10,",
        "2024-03-18,Standing Desk,Möbel,Süd,2400.00,6,0,4.6",
        "2024-03-22,Widget Pro,Elektronik,Nord,1920.00,38,5,4.3",
        "2024-03-25,Gadget Mini,Elektronik,Ost,1240.00,34,8,3.9",
        "2024-03-28,Kabelset,Zubehör,Nord,280.00,70,0,4.0",
        "2024-04-02,Standing Desk,Möbel,West,4000.00,10,0,4.8",
        "2024-04-05,Widget Pro,Elektronik,Süd,,28,12,3.5",
        "2024-04-08,Office Chair,Möbel,Ost,1335.00,15,5,4.2",
        "2024-04-12,Gadget Mini,Elektronik,Nord,2730.00,76,3,4.4",
        "2024-04-15,Kabelset,Zubehör,Süd,640.00,160,0,3.8",
        "2024-04-18,Standing Desk,Möbel,Nord,3600.00,9,8,4.5",
    ]
    with open(beispiel_csv, "w") as f:
        f.write("\n".join(daten))
    print(f"✓ Beispieldatei '{beispiel_csv}' erzeugt (30 Zeilen, 2 mit fehlenden Werten)")
else:
    print(f"✓ Beispieldatei '{beispiel_csv}' bereits vorhanden")

In [ ]:
# ── Schnelltest (ohne Gradio) ──
# Kommentar entfernen und ausführen, um den Agenten direkt zu testen:

#result = run_analysis(beispiel_csv)
#print(result)

## 9. Gradio-Interface

[Gradio](https://gradio.app/) gibt uns mit wenigen Zeilen ein Web-Interface. 
Der Nutzer lädt eine Datei hoch und bekommt die Analyse als Markdown-Bericht.

**Warum Gradio?**
- ~20 Zeilen für ein funktionales UI
- `share=True` erzeugt einen öffentlichen Link (72h gültig)
- Läuft direkt im Jupyter Notebook
- Deutlich schneller als Flask/FastAPI für Prototypen

In [ ]:
import gradio as gr


def gradio_analyze(file):
    """
    Wrapper für das Gradio-Interface.
    Gradio übergibt hochgeladene Dateien als temporäre Pfade.
    """
    if file is None:
        return "⚠️ Bitte lade eine CSV- oder Excel-Datei hoch."
    
    file_path = file.name if hasattr(file, "name") else str(file)
    
    try:
        return run_analysis(file_path)
    except Exception as e:
        return f"❌ **Fehler bei der Analyse:**\n\n`{type(e).__name__}: {e}`"


# ══════════════════════════════════════════════════════════════
# Gradio-App definieren
# ══════════════════════════════════════════════════════════════
demo = gr.Interface(
    fn=gradio_analyze,
    
    # ── Eingabe ──
    inputs=gr.File(
        label="📁 CSV- oder Excel-Datei hochladen",
        file_types=[".csv", ".xlsx", ".xls", ".tsv"],
    ),
    
    # ── Ausgabe ──
    outputs=gr.Markdown(
        label="📊 Analyseergebnis",
    ),
    
    # ── UI-Einstellungen ──
    title="🔬 Datenanalyse-Agent (Stufe 1)",
    description=(
        "Lade eine CSV- oder Excel-Datei hoch. "
        "Der Agent analysiert die Daten automatisch: "
        "Struktur erkennen → Analysen durchführen → "
        "Visualisierungen erstellen → Erkenntnisse zusammenfassen."
    ),
    
    # Beispieldatei als Schnellstart
    examples=[[beispiel_csv]] if os.path.exists(beispiel_csv) else None,
    flagging_mode="never",
)

print("✓ Gradio-Interface definiert")
print("  Starte mit der nächsten Zelle: demo.launch()")

### Interface starten

| Modus | Code | Beschreibung |
|-------|------|--------------|
| Lokal | `demo.launch()` | Nur auf diesem Rechner |
| Netzwerk | `demo.launch(server_name="0.0.0.0")` | Im lokalen Netzwerk erreichbar |
| Öffentlich | `demo.launch(share=True)` | Temporärer öffentlicher Link (72h) |

In [ ]:
# Interface starten
demo.launch(
    # share=True,              # Öffentlichen Link erzeugen
    # server_name="0.0.0.0",   # Im Netzwerk erreichbar
    # server_port=7860,        # Port festlegen
)

---

## 10. Ausblick: Stufe 2 → Reflexionsschleife

In Stufe 2 erweitern wir den Agenten um einen **Qualitäts-Agenten**, 
der die Ergebnisse des Analysten bewertet (Reflection Pattern):

```
analyst → quality_checker ──(Score < 0.7)──→ analyst   (max. 3x)
                           └──(Score ≥ 0.7)──→ report_generator → END
```

**Neue Konzepte:**
- **Zyklen mit Abbruchbedingung** – verhindert Endlosschleifen
- **Zwei LLMs mit verschiedenen Rollen** – Analyst ≠ Reviewer
- **Qualitäts-Score als Routing-Signal** – konditionale Kante basiert auf LLM-Bewertung

**Stufe 3:** Human-in-the-Loop + Checkpointing

**Stufe 4:** Report-Generator + Produktionshärtung (Error-Handling, Timeouts, Logging)